# Integrator Disorders

Disorders of the **neural integrator** (NPH/INC) and **velocity storage** (VN / nodulus).
Both are bilateral leaky integrators with a slow null-point adaptation state.

## Architecture

```
NI state:  [x_L(3) | x_R(3) | x_null(3)]   — NPH left/right pops + adapted null
VS state:  [x_L(3) | x_R(3) | x_null(3)]   — VN  left/right pops + adapted null

Net NI:  x_L − x_R  (eye position command, deg)
Net VS:  x_L − x_R  (head velocity estimate, deg/s)

NI dynamics:  d(x_net)/dt = −(1/τ_i) · (x_net − x_null) + u_vel
NI null:      dx_null/dt  =  (x_net − x_null) / τ_ni_adapt

VS dynamics:  d(x_net)/dt = −(1/τ_vs) · (x_net − x_null) + B @ u_lin
VS null:      dx_null/dt  =  (w_est  − x_null) / τ_vs_adapt
```

## Null-point adaptation — rebound nystagmus mechanism

The null slowly tracks the integrator's current output.  
During sustained eccentric gaze (NI) or sustained VOR/OKAN (VS): `x_null` builds toward the current value.  
When the sustained input stops: the integrator leaks toward `x_null` (not toward 0) → **rebound** in the original direction.

**τ_ni_adapt** (default 20 s): ~half-period rebound after 10–20 s eccentric gaze.  
**τ_vs_adapt** (default 600 s): negligible in normal demos; reduce to ~30–60 s for PAN.

## Literature (confidence: ✅ high | ⚠️ moderate | ❓ uncertain)

| Claim | Source | Confidence |
|-------|--------|------------|
| Rebound nystagmus after eccentric gaze — cerebellar sign | Zee et al. (1980) *Brain* | ✅ |
| Floccular Purkinje cells discharge during rebound | Hess et al. (1985) | ⚠️ |
| NI leakiness (τ_i shortened) → gaze-evoked nystagmus | Cannon & Robinson (1985) *Biol Cybern* | ✅ |
| Nodulus ablation → prolonged OKAN, PAN-like oscillation | Cohen et al. (1992) *J Neurophysiol* | ✅ |
| PAN period ~2 min in humans | Leigh et al. (1981) *Neurology* | ✅ |
| PAN: slow centripetal drift in darkness | Baloh et al. (1976) | ⚠️ |
| Square-wave jerks — NI instability with short τ_i | Dell'Osso et al. (1975) | ⚠️ |
| Infantile nystagmus syndrome — foveation period model | Jacobs et al. (2009) *J Vis* | ⚠️ |

## Conditions catalog

| Condition | Simulatable now? | Key parameters | Notes |
|-----------|-----------------|----------------|-------|
| **Gaze-evoked nystagmus** | ✅ | `τ_i ↓` (e.g. 2 s) | Centripetal drift during eccentric gaze |
| **Rebound nystagmus** | ✅ | `τ_ni_adapt` default 20 s | After sustained eccentric gaze |
| **Bruns nystagmus** | ✅ | `τ_i ↓` + small `τ_ni_adapt` | Gaze-evoked + rebound combined |
| **PAN (periodic alternating)** | ⚠️ partial | `τ_vs_adapt ↓` (~30 s) | VS null oscillation; needs tuning |
| **Square-wave jerks** | ❌ not yet | NI instability + fixation | Needs position-error fixation loop |
| **Congenital nystagmus** | ❌ not yet | Foveation instability | Needs sensorimotor tuning model |

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt

from oculomotor.sim.simulator import (
    PARAMS_DEFAULT, with_brain, with_sensory, simulate,
    _IDX_VS, _IDX_VS_L, _IDX_VS_R, _IDX_VS_NULL,
    _IDX_NI, _IDX_NI_L, _IDX_NI_R, _IDX_NI_NULL,
    _IDX_VIS, _IDX_SG,
)
from oculomotor import __version__
from oculomotor.analysis import (
    vs_net as _vs_net3, ni_net as _ni_net3,
    vs_null as _vs_null3, ni_null as _ni_null3,
    extract_burst as _extract_burst3,
    extract_spv, ax_fmt,
)

print(f'oculomotor {__version__}')
THETA = PARAMS_DEFAULT
print(f'τ_i={THETA.brain.tau_i}s  τ_ni_adapt={THETA.brain.tau_ni_adapt}s')
print(f'τ_vs={THETA.brain.tau_vs}s  τ_vs_adapt={THETA.brain.tau_vs_adapt}s')

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 9

SR  = 200.0
DT  = 1.0 / SR

# ── Yaw-only wrappers (notebook convention: scalar time series) ────────────────
# Full 3D signals available as _vs_net3(states), _ni_net3(states), etc.
def ni_net(states):   return _ni_net3(states)[:, 0]
def ni_null(states):  return _ni_null3(states)[:, 0]
def vs_net(states):   return _vs_net3(states)[:, 0]
def vs_null(states):  return _vs_null3(states)[:, 0]
def extract_burst(states, theta): return _extract_burst3(states, theta)[:, 0]
def eye_pos(states):  return np.array(states.plant[:, 0])
def eye_vel(states):  return np.gradient(eye_pos(states), DT)

def slow_phase(t, ev, burst, threshold=10.0, margin_s=0.05):
    return extract_spv(t, ev, burst, burst_threshold=threshold, margin_s=margin_s)

---
## 1. Gaze-Evoked Nystagmus — Leaky NI

With shortened `τ_i`, the NI leaks back toward zero during sustained eccentric gaze.  
Eye drifts centripetally → corrective fast phases → gaze-evoked nystagmus beating in direction of gaze.

**Anatomy:** Flocculus / paraflocculus lesion loses its clamping of the NI → τ_i effectively shorter.  
Also seen with brainstem NPH lesions.

**Parameters:** `τ_i = 2.0 s` (healthy: 25 s)

**Expected:**
- At centre gaze: no nystagmus
- At 20° eccentric gaze: drift toward centre + corrective fast phases
- SPV scales with eccentricity (Alexander's law)

**Ref:** Cannon & Robinson (1985) *Biol Cybern* ✅

In [ ]:
%%time
THETA_GEN = with_brain(THETA, tau_i=2.0)   # leaky NI

# Protocol: look 20° right, hold, return to centre
DUR_HOLD = 15.0; DUR_REST = 5.0
t_gn = jnp.arange(0.0, DUR_HOLD + DUR_REST, DT)
T_gn = len(t_gn)

# Target jumps to 20° at t=0, returns to 0° at t=DUR_HOLD
target_x = jnp.where(t_gn < DUR_HOLD, jnp.tan(jnp.radians(20.0)), 0.0)
pt3_gn   = jnp.stack([target_x, jnp.zeros(T_gn), jnp.ones(T_gn)], axis=1)

st_gn_h = simulate(THETA,     t_gn, p_target_array=pt3_gn,
                   scene_present_array=jnp.ones(T_gn),
                   max_steps=int((DUR_HOLD+DUR_REST)/0.001)+500, return_states=True)
st_gn_p = simulate(THETA_GEN, t_gn, p_target_array=pt3_gn,
                   scene_present_array=jnp.ones(T_gn),
                   max_steps=int((DUR_HOLD+DUR_REST)/0.001)+500, return_states=True)

t_gn_np = np.array(t_gn)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 8), sharex=True)
fig.suptitle('Gaze-Evoked Nystagmus — Leaky NI (τ_i: 25 s → 2 s)',
             fontsize=10, fontweight='bold')

C_h = '#2166ac'; C_p = '#d6604d'

axes[0].plot(t_gn_np, eye_pos(st_gn_h), color=C_h, lw=1.5, label=f'Healthy τ_i={THETA.brain.tau_i}s')
axes[0].plot(t_gn_np, eye_pos(st_gn_p), color=C_p, lw=1.5, label=f'GEN τ_i={THETA_GEN.brain.tau_i}s')
axes[0].axvline(DUR_HOLD, color='k', lw=0.7, ls='--', alpha=0.4)
axes[0].axhline(20, color='gray', lw=0.7, ls=':', alpha=0.5, label='Target 20°')
ax_fmt(axes[0], ylabel='Eye pos (deg)')
axes[0].legend(fontsize=7)
axes[0].set_title('Eye position — GEN: centripetal drift during eccentric hold')

axes[1].plot(t_gn_np, eye_vel(st_gn_h), color=C_h, lw=1.0, alpha=0.6, label='Healthy')
axes[1].plot(t_gn_np, eye_vel(st_gn_p), color=C_p, lw=1.0, alpha=0.6, label='GEN')
axes[1].axvline(DUR_HOLD, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(axes[1], ylabel='Eye vel (deg/s)', ylim=(-300, 300))
axes[1].legend(fontsize=7)
axes[1].set_title('Eye velocity — corrective saccades during eccentric gaze')

axes[2].plot(t_gn_np, ni_net(st_gn_h), color=C_h, lw=1.5, label='NI net healthy')
axes[2].plot(t_gn_np, ni_net(st_gn_p), color=C_p, lw=1.5, label='NI net GEN')
axes[2].plot(t_gn_np, ni_null(st_gn_p), color=C_p, lw=1.0, ls='--', alpha=0.6, label='NI null GEN')
axes[2].axvline(DUR_HOLD, color='k', lw=0.7, ls='--', alpha=0.4)
ax_fmt(axes[2], ylabel='NI state (deg)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('NI state — leaky NI cannot hold eccentric position; null adapts slowly')

fig.tight_layout()
plt.show()

---
## 2. Rebound Nystagmus — NI Null Adaptation

After sustained eccentric gaze, the NI null (`x_null`) has slowly drifted toward the eccentric position.  
On return to centre, the NI leaks toward `x_null` (not toward 0) → eye drifts toward former eccentric position  
→ slow phase eccentric → fast phase centripetal → **nystagmus beats toward centre** ("rebound").

Rebound nystagmus is a hallmark of **cerebellar disease** (flocculus/paraflocculus).  
It coexists with GEN: combined picture = Bruns nystagmus.

**Parameters:** default `τ_i=25 s`, `τ_ni_adapt=20 s` (null builds over ~20 s of eccentric gaze)

**Protocol:** Hold 20° right for 20 s, then jump back to 0°.  

**Expected:** Brief rebound nystagmus beating LEFT after returning to centre.

**Ref:** Zee et al. (1980) *Brain* ✅; Hood & Leech (1974) *J Neurol Neurosurg Psychiatry* ⚠️

In [ ]:
%%time
# Protocol: hold 20° eccentric for HOLD_DUR, then return to 0°
HOLD_DUR  = 20.0   # s — long enough for null to build
POST_DUR  = 15.0   # s — watch rebound decay
t_rb  = jnp.arange(0.0, HOLD_DUR + POST_DUR, DT)
T_rb  = len(t_rb)
t_rb_np = np.array(t_rb)

target_x_rb = jnp.where(t_rb < HOLD_DUR, jnp.tan(jnp.radians(20.0)), 0.0)
pt3_rb      = jnp.stack([target_x_rb, jnp.zeros(T_rb), jnp.ones(T_rb)], axis=1)

max_s_rb = int((HOLD_DUR + POST_DUR) / 0.001) + 1000

# Compare: no null adaptation (large tau_ni_adapt) vs default
THETA_NO_NULL = with_brain(THETA, tau_ni_adapt=1e6)  # null frozen → no rebound

st_rb_null = simulate(THETA,         t_rb, p_target_array=pt3_rb,
                      scene_present_array=jnp.ones(T_rb),
                      max_steps=max_s_rb, return_states=True)
st_rb_nonull = simulate(THETA_NO_NULL, t_rb, p_target_array=pt3_rb,
                        scene_present_array=jnp.ones(T_rb),
                        max_steps=max_s_rb, return_states=True)

burst_rb     = extract_burst(st_rb_null, THETA)
burst_rb_nn  = extract_burst(st_rb_nonull, THETA_NO_NULL)
spv_rb       = slow_phase(t_rb_np, eye_vel(st_rb_null),   burst_rb)
spv_rb_nn    = slow_phase(t_rb_np, eye_vel(st_rb_nonull), burst_rb_nn)

null_at_return = ni_null(st_rb_null)[int(HOLD_DUR / DT)]
print(f'NI null at t={HOLD_DUR}s (return to centre): {null_at_return:.1f} deg')
print(f'Expected rebound slow phase: rightward (positive) → beats LEFT')
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle(f'Rebound Nystagmus — NI Null Adaptation (τ_ni_adapt = {THETA.brain.tau_ni_adapt:.0f} s)',
             fontsize=10, fontweight='bold')

C_null   = '#2166ac'
C_nonull = '#999999'

axes[0].plot(t_rb_np, eye_pos(st_rb_null),   color=C_null,   lw=1.5, label=f'τ_null={THETA.brain.tau_ni_adapt:.0f}s (rebound)')
axes[0].plot(t_rb_np, eye_pos(st_rb_nonull), color=C_nonull, lw=1.5, ls='--', label='τ_null=∞ (no rebound)')
axes[0].axvline(HOLD_DUR, color='k', lw=0.7, ls='--', alpha=0.5, label='Return to centre')
axes[0].axhline(20, color='gray', lw=0.7, ls=':', alpha=0.4)
ax_fmt(axes[0], ylabel='Eye pos (deg)')
axes[0].legend(fontsize=7)
axes[0].set_title('Eye position — after return, eye drifts right (toward former target)')

axes[1].plot(t_rb_np, spv_rb,    color=C_null,   lw=2.0, label='SPV (with null)')
axes[1].plot(t_rb_np, spv_rb_nn, color=C_nonull, lw=1.5, ls='--', label='SPV (no null)')
axes[1].axvline(HOLD_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[1], ylabel='SPV (deg/s)', ylim=(-30, 30))
axes[1].legend(fontsize=7)
axes[1].set_title('SPV — positive after return = rightward slow phase → nystagmus beats LEFT (rebound ✓)')

axes[2].plot(t_rb_np, ni_net(st_rb_null),  color=C_null,   lw=1.5, label='NI net')
axes[2].plot(t_rb_np, ni_null(st_rb_null), color='#e08214', lw=1.5, ls='-', label='NI null')
axes[2].axvline(HOLD_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[2], ylabel='NI state (deg)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('NI state — null persists after return; NI leaks toward null → rightward drift')

fig.tight_layout()
plt.show()

---
## 3. Bruns Nystagmus — GEN + Rebound Combined

Bruns nystagmus is seen with cerebellopontine angle tumors (acoustic neuroma):  
- **Ipsilesional gaze:** large-amplitude, slow-frequency nystagmus (GEN from VS imbalance + leaky NI)  
- **Contraversive gaze:** small-amplitude, high-frequency nystagmus (pure rebound)

Model: combine `τ_i ↓` (leaky NI) + `τ_ni_adapt` (null adaptation) + unilateral hypofunction.

**Parameters:** `τ_i = 3.0 s`, `τ_ni_adapt = 15.0 s`, left UVH

**Expected:**
- Spontaneous rightward nystagmus at rest (from UVH)
- Large nystagmus during leftward (ipsilesional) gaze  
- Smaller, faster nystagmus during rightward (contraversive) gaze + rebound on return

**Ref:** Bruns (1902); reviewed in Leigh & Zee (2015) ✅

In [ ]:
%%time
from oculomotor.sim.simulator import with_uvh

# Bruns: leaky NI + active rebound + left UVH
THETA_BRUNS = with_uvh(
    with_brain(THETA, tau_i=3.0, tau_ni_adapt=15.0),
    side='left', canal_gain_frac=0.1
)

# Protocol: 0° → +20° (rightward, contraversive) → 0° → −20° (leftward, ipsilesional) → 0°
SEG = 12.0  # s per segment
T_TOTAL = 4 * SEG
t_br    = jnp.arange(0.0, T_TOTAL, DT)
T_br    = len(t_br)
t_br_np = np.array(t_br)

def _target(t_arr, segs):
    """Step through target positions."""
    tx = jnp.zeros_like(t_arr)
    for i, (t_start, deg) in enumerate(segs):
        tx = jnp.where(t_arr >= t_start, jnp.tan(jnp.radians(deg)), tx)
    return tx

tx_br = _target(t_br, [(0, 0), (SEG, 20), (2*SEG, 0), (3*SEG, -20)])
pt3_br = jnp.stack([tx_br, jnp.zeros(T_br), jnp.ones(T_br)], axis=1)

max_br = int(T_TOTAL / 0.001) + 2000
st_br = simulate(THETA_BRUNS, t_br, p_target_array=pt3_br,
                 scene_present_array=jnp.ones(T_br),
                 max_steps=max_br, return_states=True)

burst_br = extract_burst(st_br, THETA_BRUNS)
spv_br   = slow_phase(t_br_np, eye_vel(st_br), burst_br)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Bruns Nystagmus — Leaky NI + Rebound + Left UVH',
             fontsize=10, fontweight='bold')

C_br = '#35978f'

axes[0].plot(t_br_np, eye_pos(st_br), color=C_br, lw=1.2, label='Eye pos')
# Target reference
target_deg = np.degrees(np.arctan(np.array(tx_br)))
axes[0].plot(t_br_np, target_deg, color='gray', lw=0.8, ls='--', label='Target')
for seg in range(1, 4):
    axes[0].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[0], ylabel='Eye pos (deg)')
axes[0].legend(fontsize=7)
axes[0].set_title('Eye position — large drift during ipsilesional (leftward) gaze')

axes[1].plot(t_br_np, spv_br, color=C_br, lw=2.0)
for seg in range(1, 4):
    axes[1].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[1], ylabel='SPV (deg/s)', ylim=(-60, 60))
axes[1].set_title('SPV — spontaneous at rest; large ipsilesional; rebound on return')

axes[2].plot(t_br_np, ni_net(st_br),  color=C_br,   lw=1.5, label='NI net')
axes[2].plot(t_br_np, ni_null(st_br), color='#e08214', lw=1.5, ls='--', label='NI null')
for seg in range(1, 4):
    axes[2].axvline(seg*SEG, color='k', lw=0.5, ls='--', alpha=0.3)
ax_fmt(axes[2], ylabel='NI state (deg)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('NI state — null follows eye position; persists after each gaze shift → rebound')

labels = ['centre', '+20° contra', 'centre', '−20° ipsi']
for i, lbl in enumerate(labels):
    axes[0].text(i*SEG + SEG/2, -30, lbl, ha='center', fontsize=7, color='gray')

fig.tight_layout()
plt.show()

---
## 4. VS Null Adaptation — Extended OKAN / Post-Rotatory After-Effect

The VS null (`x_null_vs`) slowly tracks `w_est`.  
After sustained OKAN, `x_null_vs` has partially built up.  
When OKAN decays, the VS leaks toward `x_null_vs` — extending the effective TC.  
With small `τ_vs_adapt`, the VS can exhibit an after-rebound.

**Parameters:** `τ_vs_adapt = 60 s` (vs default 600 s → reduced to make effect visible)

**Protocol:** OKN 30 deg/s for 30 s, then lights off (OKAN).  

**Note on PAN:** For periodic alternating nystagmus (period ~2 min), τ_vs_adapt needs to be tuned  
alongside the coupling sign. The current excitatory null coupling prolongs OKAN; PAN requires  
the null to oscillate — future tuning exercise.

**Ref:** Cohen et al. (1992) *J Neurophysiol* (nodulus ablation) ✅

In [ ]:
%%time
THETA_VS_ADAPT = with_brain(THETA, tau_vs_adapt=60.0)   # faster null → visible TC extension

ON_DUR = 30.0; OFF_DUR = 60.0
t_vn   = jnp.arange(0.0, ON_DUR + OFF_DUR, DT)
T_vn   = len(t_vn)
t_vn_np = np.array(t_vn)

v_sc  = jnp.zeros((T_vn, 3)).at[:, 0].set(jnp.where(t_vn < ON_DUR, 30.0, 0.0))
sc_pr = jnp.where(t_vn < ON_DUR, 1.0, 0.0)

max_vn = int((ON_DUR + OFF_DUR) / 0.001) + 1000
st_vn_h = simulate(THETA,          t_vn, v_scene_array=v_sc, scene_present_array=sc_pr,
                   target_present_array=jnp.zeros(T_vn), max_steps=max_vn, return_states=True)
st_vn_a = simulate(THETA_VS_ADAPT, t_vn, v_scene_array=v_sc, scene_present_array=sc_pr,
                   target_present_array=jnp.zeros(T_vn), max_steps=max_vn, return_states=True)

burst_vn_h = extract_burst(st_vn_h, THETA)
burst_vn_a = extract_burst(st_vn_a, THETA_VS_ADAPT)
spv_vn_h   = slow_phase(t_vn_np, eye_vel(st_vn_h), burst_vn_h)
spv_vn_a   = slow_phase(t_vn_np, eye_vel(st_vn_a), burst_vn_a)
print('Done.')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('VS Null Adaptation — Extended OKAN (τ_vs_adapt = 60 s vs 600 s)',
             fontsize=10, fontweight='bold')

C_h = '#2166ac'; C_a = '#d6604d'

axes[0].plot(t_vn_np, spv_vn_h, color=C_h, lw=2.0, label=f'Default τ_vs_adapt={THETA.brain.tau_vs_adapt:.0f}s')
axes[0].plot(t_vn_np, spv_vn_a, color=C_a, lw=2.0, label=f'VS adapt τ_vs_adapt={THETA_VS_ADAPT.brain.tau_vs_adapt:.0f}s')
axes[0].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5, label='Lights off')
axes[0].axhline(30, color='gray', lw=0.7, ls=':', alpha=0.4)
ax_fmt(axes[0], ylabel='SPV (deg/s)')
axes[0].legend(fontsize=7)
axes[0].set_title('OKN + OKAN — faster VS null extends OKAN')

axes[1].plot(t_vn_np, -vs_net(st_vn_h), color=C_h, lw=1.5, label='VS net (healthy)')
axes[1].plot(t_vn_np, -vs_net(st_vn_a), color=C_a, lw=1.5, label='VS net (adapted)')
axes[1].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[1], ylabel='VS net (deg/s)')
axes[1].legend(fontsize=7)
axes[1].set_title('VS state — same charging; decay toward null (not 0) extends OKAN')

axes[2].plot(t_vn_np, -vs_null(st_vn_h), color=C_h, lw=1.0, ls='--', alpha=0.5, label='VS null (healthy, barely visible)')
axes[2].plot(t_vn_np, -vs_null(st_vn_a), color=C_a, lw=2.0, label='VS null (adapted)')
axes[2].axvline(ON_DUR, color='k', lw=0.7, ls='--', alpha=0.5)
ax_fmt(axes[2], ylabel='VS null (deg/s)', xlabel='Time (s)')
axes[2].legend(fontsize=7)
axes[2].set_title('VS null adaptation — builds during OKN; residual prolongs OKAN')

fig.tight_layout()
plt.show()

---
## 5. Future: Periodic Alternating Nystagmus (PAN)

**Mechanism (proposed):**  
The cerebellar nodulus adapts to ongoing spontaneous nystagmus, attempting to cancel it.  
With appropriate VS null dynamics (τ_vs_adapt ~30–60 s, inhibitory coupling),  
the null overshoots → nystagmus reverses → null adapts again → ~2-min oscillation.

**Current model limitation:** The null adaptation here is EXCITATORY (null → sustains the state).  
PAN requires INHIBITORY coupling (null → opposes the state) — a sign flip in the null→b_eff coupling.  
This would change VS dynamics from `b_eff = b ± x_null/2` to `b_eff = b ∓ x_null/2`.  

**Implementation plan:**
- Add `sign_vs_null: float = +1.0` parameter to BrainParams (default +1 = excitatory/sustaining)
- Set `sign_vs_null = -1.0` for PAN modelling (inhibitory/anti-drift)
- Tune `τ_vs_adapt` + initial VN imbalance → spontaneous nystagmus → PAN oscillation

**Ref:** Cohen et al. (1988, 1992); Leigh et al. (1981) ✅

In [ ]:
print('PAN simulation pending sign_vs_null parameter — see section description above.')